# 📘 Notebook 13: Meet a Real Warehouse

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module E: A Real Warehouse, End to End · Notebook 1 of 6 in this module · (13 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Download a real data warehouse and take an **inventory** of its tables
- Tell **fact** tables from **dimension** tables, and state the **grain** of a fact table with evidence
- Measure **missing values** and decide which ones matter
- Check **referential integrity**: does every key in a fact table find its partner?
- Spot **personal data** and apply the principle of data minimisation before any analysis

> **Prerequisites:** Notebooks 01, 02 and 04 (keys, joins, the star schema). Loading tables with pandas, from *From Python to Deep Learning and Fine-Tuning*, Notebook 06.
>
> 🔭 **Why this notebook matters:** every warehouse in this course so far was invented by us, so it held no surprises. Real data always does. Before a single chart or model, a professional spends time simply *getting to know* the data: what is there, how it is shaped, what is missing, what does not join, and what should not be there at all. This step is unglamorous, and it is where most analytical mistakes are prevented. In this module we take one real warehouse through the whole course: profile it, build it, reshape it, explore it with Spark, learn from it, and measure what scale really means for it.

> 🗃️ **The data.** This module works with **AdventureWorks**, the sample data warehouse that Microsoft publishes for learning SQL Server: the online sales of an imaginary bicycle company, with its customers, products and dates. It is open data (MIT licence) and much closer to a real warehouse than anything we could invent. The cell below downloads it from Microsoft's GitHub repository (about 17 MB, once per session) and saves each table as an ordinary CSV file.

In [ ]:
import io
import os
import re
import urllib.request
import zipfile

import pandas as pd

AW_URL = ("https://github.com/microsoft/sql-server-samples/releases/download/"
          "adventureworks/AdventureWorksDW-data-warehouse-install-script.zip")
AW_TABLES = ["FactInternetSales", "DimCustomer", "DimGeography", "DimSalesTerritory", "DimProduct",
             "DimProductSubcategory", "DimProductCategory", "DimDate", "DimPromotion", "DimCurrency"]


def fetch_adventureworks(folder="adventureworks_dw"):
    """Download Microsoft's AdventureWorks sample warehouse once, and save each table as a CSV file with a header."""
    if os.path.isdir(folder):
        return folder
    archive = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(AW_URL).read()))
    script = archive.read("instawdbdw.sql")
    script = script.decode("utf-16" if script[:2] in (b"\xff\xfe", b"\xfe\xff") else "utf-8-sig")
    os.makedirs(folder)
    for table in AW_TABLES:
        # The column names are not in the data files: they are in the CREATE TABLE statements of the install script.
        body = re.search(r"CREATE TABLE \[dbo\]\.\[" + table + r"\]\((.*?)\n\)", script, re.S).group(1)
        columns = re.findall(r"^\s*\[(\w+)\]", body, re.M)
        text = archive.read(table + ".csv").decode("utf-8-sig")
        df = pd.read_csv(io.StringIO(text), sep="|", header=None, names=columns, dtype=str, keep_default_na=False)
        df = df.drop(columns=[c for c in ("LargePhoto", "SalesTerritoryImage") if c in df])   # pictures as hex text
        df.replace("", None).to_csv(f"{folder}/{table}.csv", index=False)
    return folder


def load(name, folder=None):
    """Read one table into pandas. Other exports of this warehouse are sometimes saved as Latin-1, so try both."""
    path = f"{folder or DATA}/{name}.csv"
    try:
        return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", low_memory=False)


DATA = fetch_adventureworks()
print("Tables:", sorted(f[:-4] for f in os.listdir(DATA)))

## 1. An inventory

### Intuition first
Imagine being handed the keys to a stranger's warehouse. You would not start by rearranging the shelves: you would walk the aisles and write down what is there. With data, the walk takes a few lines of code.

The table names follow the convention of Notebook 04. Names beginning with **Fact** hold events that are counted and summed, here the sales made on the company's website. Names beginning with **Dim** describe what the events are about: customers, products, dates, places.

In [ ]:
rows = []
for name in sorted(f[:-4] for f in os.listdir(DATA)):
    t = load(name)
    rows.append({"table": name, "kind": "fact" if name.startswith("Fact") else "dimension",
                 "rows": len(t), "columns": t.shape[1],
                 "memory_MB": round(t.memory_usage(deep=True).sum() / 1e6, 1)})
inventory = pd.DataFrame(rows).sort_values("rows", ascending=False)
inventory

In [ ]:
print("Rows in all tables:", inventory["rows"].sum())
print("Memory in pandas:   ", round(inventory["memory_MB"].sum()), "MB")

Fewer than a hundred thousand rows and a few tens of megabytes. That is small: it fits many times over in the memory of any laptop. Keep the number in mind. A recurring question of this module is what would change if it were a thousand times larger, and the honest answer is not always "use Spark". Notebook 18 measures it.

## 2. The fact table and its grain

### Intuition first
The **grain** of a fact table is the answer to *"what is one row?"*. It is the most important sentence in a warehouse's documentation, because every sum and every count depends on it. Count the rows of a table whose grain you have misunderstood, and you will report the wrong number with complete confidence.

Look at the main fact table first.

In [ ]:
sales = load("FactInternetSales")
sales.head()

In [ ]:
sales.dtypes.value_counts()

The column names suggest a hypothesis: one row is **one line of one sales order**. A hypothesis about grain can be tested. If it is true, the pair (`SalesOrderNumber`, `SalesOrderLineNumber`) never repeats.

In [ ]:
pairs = sales[["SalesOrderNumber", "SalesOrderLineNumber"]].drop_duplicates()
print("Rows:                 ", len(sales))
print("Distinct (order, line):", len(pairs))
print("Distinct orders:       ", sales["SalesOrderNumber"].nunique())
print("Distinct customers:    ", sales["CustomerKey"].nunique())
print("Distinct products:     ", sales["ProductKey"].nunique())

The hypothesis holds: the pair is unique, so it can serve as the table's key. An order has on average a little over two lines.

Now the **measures**, the numbers that will be summed.

In [ ]:
measures = ["OrderQuantity", "UnitPrice", "SalesAmount", "TotalProductCost", "TaxAmt", "Freight"]
sales[measures].describe().round(2)

Two things deserve a note. `OrderQuantity` is always 1: in this shop every line is a single item, so counting lines and summing quantities give the same answer. And the median sale is a few tens of currency units while the maximum is in the thousands: a few expensive bicycles among many cheap accessories. A distribution like that will matter in Notebook 16.

## 3. Missing values

### Intuition first
An empty cell can mean three different things: *the value does not apply* (a customer with no middle name), *the value was never recorded* (a lost delivery note), or *the value exists but the export lost it*. Code cannot tell them apart. It can only find them, and then a person must decide.

In [ ]:
def missing_report(df):
    share = df.isna().mean().mul(100).round(1)
    return share[share > 0].sort_values(ascending=False)

missing_report(sales)

In [ ]:
customers = load("DimCustomer")
missing_report(customers)

Two columns of the fact table are entirely empty: they carry no information and can be dropped. In the customer table, the missing titles, suffixes and second address lines simply *do not apply* to most people. None of these is a problem for analysis, but each must be a decision, written down, and not an accident.

## 4. Do the keys join? Referential integrity

### Intuition first
A fact row points to its dimensions through keys. If a `CustomerKey` in the fact table has no partner in `DimCustomer`, every inner join that involves customers will silently drop that sale. The total will be wrong and nothing will warn you. So we check, for each key, which share of fact rows finds a partner.

In [ ]:
def key_match(fact, fact_col, dim_name, dim_col=None):
    dim = load(dim_name)
    share = fact[fact_col].isin(dim[dim_col or fact_col]).mean() * 100
    print(f"{fact_col:>18} -> {dim_name:<18} {share:6.2f} % of rows have a partner")
    return share

for col, dim, dcol in [("CustomerKey", "DimCustomer", None), ("ProductKey", "DimProduct", None),
                       ("SalesTerritoryKey", "DimSalesTerritory", None), ("PromotionKey", "DimPromotion", None),
                       ("CurrencyKey", "DimCurrency", None), ("OrderDateKey", "DimDate", "DateKey"),
                       ("ShipDateKey", "DimDate", "DateKey"), ("DueDateKey", "DimDate", "DateKey")]:
    key_match(sales, col, dim, dcol)

Every key finds its partner. That is what a well-built warehouse looks like, and it is worth checking every time, because a copy of a warehouse is not always as well built as the original. (If you are given another export of these tables, run this cell on it first.)

Notice one thing in the last three lines: the same `DimDate` table answers for three different dates of an order. A dimension used in several roles is called a **role-playing dimension**. We will meet it again in Notebook 14.

The dimensions point to each other too. A product belongs to a subcategory, which belongs to a category:

In [ ]:
products = load("DimProduct")
subcats = load("DimProductSubcategory")
print("Products:", len(products))
print("Products without a subcategory:", products["ProductSubcategoryKey"].isna().sum())
print("...of which ever sold online:", products.loc[products["ProductSubcategoryKey"].isna(), "ProductKey"]
      .isin(sales["ProductKey"]).sum())

A third of the products have no subcategory. They are parts (bolts, washers, frames for assembly) that the company keeps in its product list but never sells on the website, so they never appear in a sale. Missing, but harmless, and now we know why.

## 5. A first look at the business

The dates are stored as text. Converted to real dates, they let us see the period the data covers.

In [ ]:
sales["OrderDate"] = pd.to_datetime(sales["OrderDate"])
print("First order:", sales["OrderDate"].min().date(), "  Last order:", sales["OrderDate"].max().date())
sales.groupby(sales["OrderDate"].dt.year)["SalesAmount"].agg(["count", "sum"]).round(0)

Two warnings are hidden in that small table. The first and last years are **not complete**: a comparison of 2010 with 2011 would be meaningless. And between 2012 and 2013 the number of order lines grows many times while revenue grows much less: the shop must have started selling something new and cheap. A good first look raises questions like these. Exercise 2 answers this one.

In [ ]:
per_customer = sales.groupby("CustomerKey")["SalesOrderNumber"].nunique()
per_customer.value_counts().sort_index().head(8)

Most customers ordered once. The few who ordered many times are worth a look before any average is computed.

## 6. Personal data: keep only what the analysis needs

### Intuition first
The customer table holds names, e-mail addresses, phone numbers, street addresses and dates of birth. They are invented here, but in a real warehouse they would belong to real people, and data protection law (in Europe, the GDPR) says that personal data must be **adequate, relevant and limited to what is necessary**. Analysis of sales needs an age band, a region and an income band. It does not need anyone's name or telephone number.

The cleanest moment to apply this is now, before the data spreads into notebooks, charts and copies.

In [ ]:
identifying = ["Title", "FirstName", "MiddleName", "LastName", "Suffix", "EmailAddress", "Phone",
               "AddressLine1", "AddressLine2", "CustomerAlternateKey"]
analysis_customers = customers.drop(columns=identifying)
print("Columns kept:", list(analysis_customers.columns))

The `CustomerKey` stays: it is a meaningless number that links a customer's sales without saying who they are. That is a **pseudonym**, the idea Notebook 12 used in its pipeline. The date of birth is also personal; Exercise 4 replaces it with something less revealing.

## 7. Where we stand

| The question | How we answered it |
|---|---|
| What is in the warehouse? | an inventory: tables, rows, columns, memory |
| What is one row of the fact table? | a grain hypothesis, tested on the key |
| What is missing, and does it matter? | a missing-value report, read column by column |
| Does every key join? | a match rate for every foreign key, and for links between dimensions |
| What should not be here at all? | personal data, removed before the analysis starts |

None of this produced a chart or a model, and all of it was necessary. The next notebook takes the profiled tables and builds them into a database of our own, with the keys and constraints that make the checks of this notebook permanent.

---
## ✏️ Exercises

### Exercise 1 (The grain of another table)
Dimension tables have a grain too. What is one row of `DimGeography`? Test whether `GeographyKey` is unique, and whether the pair (`City`, `PostalCode`) is.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
geo = load("DimGeography")
print("Rows:", len(geo))
print("Distinct GeographyKey:", geo["GeographyKey"].nunique())
print("Distinct (City, PostalCode):", len(geo[["City", "PostalCode"]].drop_duplicates()))
```

*GeographyKey is unique, so one row is one place. The pair (City, PostalCode) is very nearly unique too, which is what we would expect of a place.*

</details>

### Exercise 2 (What happened in 2013?)
Join the sales to the product, subcategory and category tables and show the number of order lines and the revenue **per category and year**. Which categories did the shop begin selling, and when?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
cats = load("DimProductCategory")
lines = (sales.merge(products[["ProductKey", "ProductSubcategoryKey"]], on="ProductKey")
              .merge(subcats[["ProductSubcategoryKey", "ProductCategoryKey"]], on="ProductSubcategoryKey")
              .merge(cats[["ProductCategoryKey", "EnglishProductCategoryName"]], on="ProductCategoryKey"))
lines.pivot_table(index=lines["OrderDate"].dt.year, columns="EnglishProductCategoryName",
                  values="SalesAmount", aggfunc=["count", "sum"]).round(0)
```

*Until the end of 2012 the website sold only bicycles. Accessories and clothing arrived later and, in 2013, outnumbered the bicycle lines several times over while adding much less revenue. Any trend computed over the whole period mixes two different shops.*

</details>

### Exercise 3 (A reusable integrity check)
Write a function `integrity(fact, links)` that takes the fact table and a list of `(fact_column, dimension_name, dimension_column)` and returns a DataFrame with one row per link and the columns `link` and `match_percent`. Run it on the links of Section 4.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def integrity(fact, links):
    out = []
    for col, dim_name, dim_col in links:
        dim = load(dim_name)
        out.append({"link": f"{col} -> {dim_name}.{dim_col or col}",
                    "match_percent": round(fact[col].isin(dim[dim_col or col]).mean() * 100, 2)})
    return pd.DataFrame(out)

integrity(sales, [("CustomerKey", "DimCustomer", None), ("ProductKey", "DimProduct", None),
                  ("OrderDateKey", "DimDate", "DateKey")])
```

*A table is a better record than printed lines: it can be saved, compared between two copies of the data, and pasted into a report.*

</details>

### Exercise 4 (Age instead of a birth date, a little harder)
Replace `BirthDate` in `analysis_customers` with an **age band** (`under 35`, `35-49`, `50-64`, `65 and over`), computed at the date of the last order in the data. Then drop `BirthDate` and show how many customers fall in each band.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
reference = sales["OrderDate"].max()
born = pd.to_datetime(analysis_customers["BirthDate"])
age = (reference - born).dt.days // 365
analysis_customers["AgeBand"] = pd.cut(age, bins=[0, 34, 49, 64, 200],
                                       labels=["under 35", "35-49", "50-64", "65 and over"])
analysis_customers = analysis_customers.drop(columns=["BirthDate"])
analysis_customers["AgeBand"].value_counts().sort_index()
```

*Four bands are enough for most analyses and reveal far less about a person than an exact date of birth. Computing the age at a fixed reference date, and not today, keeps the result reproducible.*

</details>

## 🔑 Key takeaways

- **Profile before you analyse**: an inventory, the grain, the missing values, the keys, the personal data.
- The **grain** of a fact table is a hypothesis you can test: its key must be unique.
- **Missing values** need a decision, not just a count: does the value not apply, or was it lost?
- Check **referential integrity** for every foreign key. An unmatched key makes joins drop rows without warning.
- A dimension that answers for several dates of one event is a **role-playing dimension**.
- Apply **data minimisation** at the start: keep pseudonymous keys and coarse bands, drop names and contact details.

**Next:** *Notebook 14: Building the Warehouse*, where these tables become a database of our own, with keys and constraints, views and indexes, and the queries an analyst would ask of it.

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*